In [1]:
import warnings

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")

FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load raw NASA CMAPSS FD001

In [2]:
col_names = (
    ["engine_id", "cycle", "setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

df = pd.read_csv(
    "../data/raw/train_FD001.txt",
    sep=r"\s+",
    header=None,
    index_col=False,
)
df = df.dropna(axis=1, how="all")
df.columns = col_names
df["engine_id"] = df["engine_id"].astype(int)
df["cycle"] = df["cycle"].astype(int)

print(f"Shape: {df.shape}  (expect (20631, 26))")
df.head(3)

Shape: (20631, 26)  (expect (20631, 26))


,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442


## 1 · Cycles per engine histogram

In [3]:
cycles_per_engine = df.groupby("engine_id")["cycle"].max()
print(f"min={cycles_per_engine.min()}  max={cycles_per_engine.max()}  "
      f"mean={cycles_per_engine.mean():.1f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cycles_per_engine, bins=20, color="#0ea5e9", edgecolor="white", linewidth=0.5)
ax.set_xlabel("Total operational cycles per engine")
ax.set_ylabel("Count of engines")
ax.set_title("FD001: Life distribution across 100 train engines\n"
             "(range ~128–362, mean ~206 cycles)")
ax.axvline(cycles_per_engine.mean(), color="#6366f1", linestyle="--",
           label=f"mean={cycles_per_engine.mean():.0f}")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "cycles_per_engine.png", dpi=120)
plt.close(fig)
print("Saved: cycles_per_engine.png")

min=128  max=362  mean=206.3


Saved: cycles_per_engine.png


## 2 · Sensor trends for 5 engines (sensors that actually change)

In [4]:
TREND_SENSORS = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11"]
SAMPLE_ENGINES = [1, 20, 40, 60, 80]

fig, axes = plt.subplots(len(TREND_SENSORS), 1, figsize=(10, 12), sharex=False)
palette = ["#0ea5e9", "#6366f1", "#10b981", "#f59e0b", "#ef4444"]

for ax, sensor in zip(axes, TREND_SENSORS):
    for eng, color in zip(SAMPLE_ENGINES, palette):
        eng_df = df[df["engine_id"] == eng].sort_values("cycle")
        ax.plot(eng_df["cycle"], eng_df[sensor],
                color=color, alpha=0.8, linewidth=1.2, label=f"Eng {eng}")
    ax.set_ylabel(sensor, fontsize=9)
    ax.tick_params(axis="both", labelsize=8)

axes[0].legend(fontsize=8, ncol=5, loc="upper right")
axes[-1].set_xlabel("Operational cycle")
fig.suptitle("Sensor trends across engine lifetime\n"
             "(visible drift confirms temporal degradation patterns)",
             fontsize=11, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "sensor_trends_5_engines.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("Saved: sensor_trends_5_engines.png")

Saved: sensor_trends_5_engines.png


## 3 · Constant sensors check (rule C40)

In [5]:
sensor_stds = {
    f"sensor_{i}": df[f"sensor_{i}"].std() for i in range(1, 22)
}
sensor_nunique = {
    f"sensor_{i}": df[f"sensor_{i}"].nunique() for i in range(1, 22)
}

# Zero std = identically constant; sensor_6 has only 2 unique values (near-constant flag)
zero_std = [s for s, std in sensor_stds.items() if std < 1e-6]
near_constant = sorted(
    set(zero_std) | {s for s, n in sensor_nunique.items() if n <= 2}
)

print(f"Exactly constant (std=0):        {zero_std}")
print(f"Near-constant (std=0 or ≤2 vals): {near_constant}")
for s in near_constant:
    print(f"  {s}: std={sensor_stds[s]:.6f}  nunique={sensor_nunique[s]}")

expected_drop = sorted([f"sensor_{i}" for i in [1, 5, 6, 10, 16, 18, 19]])
assert near_constant == expected_drop, (
    f"Mismatch! got {near_constant}, expected {expected_drop}"
)
print("\nConfirmed: all 7 drop_columns are constant or near-constant (rule C40).")

Exactly constant (std=0):        ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Near-constant (std=0 or ≤2 vals): ['sensor_1', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19', 'sensor_5', 'sensor_6']
  sensor_1: std=0.000000  nunique=1
  sensor_10: std=0.000000  nunique=1
  sensor_16: std=0.000000  nunique=1
  sensor_18: std=0.000000  nunique=1
  sensor_19: std=0.000000  nunique=1
  sensor_5: std=0.000000  nunique=1
  sensor_6: std=0.001389  nunique=2

Confirmed: all 7 drop_columns are constant or near-constant (rule C40).


## 4 · RUL distribution: raw vs capped at 125 (rule C35)

In [6]:
max_cycle = df.groupby("engine_id")["cycle"].transform("max")
rul_raw = (max_cycle - df["cycle"]).astype(float)
rul_capped = rul_raw.clip(upper=125)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(rul_raw, bins=40, color="#ef4444", edgecolor="white", linewidth=0.4)
ax1.set_title("Raw RUL (max_cycle − current_cycle)")
ax1.set_xlabel("RUL (cycles)")
ax1.set_ylabel("Row count")
ax1.text(0.65, 0.92, f"max={rul_raw.max():.0f}",
         transform=ax1.transAxes, color="#ef4444", fontsize=9)

ax2.hist(rul_capped, bins=40, color="#10b981", edgecolor="white", linewidth=0.4)
ax2.set_title("Piecewise-linear RUL cap at 125 (Zheng et al. 2017)")
ax2.set_xlabel("RUL (cycles)")
ax2.set_ylabel("Row count")
ax2.text(0.65, 0.92, f"max={rul_capped.max():.0f}",
         transform=ax2.transAxes, color="#10b981", fontsize=9)

fig.suptitle("Why cap RUL at 125? Removes the noisy healthy plateau — "
             "model capacity focuses on the degradation phase.",
             fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "rul_distribution_raw_vs_capped.png", dpi=120)
plt.close(fig)
print("Saved: rul_distribution_raw_vs_capped.png")

Saved: rul_distribution_raw_vs_capped.png


## Key Takeaways

1. **100 training engines, 20 631 rows** — life spans range 128–362 cycles (mean 206). Each engine tells a full degradation story.
2. **Sensors 2, 3, 4, 7, 11 drift monotonically** as cycles increase — these are the temporal degradation signals the LSTM will learn.
3. **7 sensors are identically constant** across all 100 engines (std < 1e-6). Dropping them leaves 17 informative features (3 settings + 14 sensors).
4. **Piecewise-linear RUL cap at 125** removes the long flat healthy plateau. Without it, the model wastes capacity predicting "engine is very healthy" — the business-critical signal is the degradation slope.
5. **Engine-id split (1–80 train, 81–100 val)** is the only valid strategy. Row-level random splits leak future cycles of training engines into validation, causing ~10–15 RMSE units of optimistic bias.